# Practice: a weekend trip planner agent

You and a friend live in Berlin and want a cheap trip this weekend: Prague or Vienna, 300 EUR for the two of you. In this workbook you build an agent that plans it, **one step at a time**. You ask the same question after every step and watch the answer get better.

| Step | What we add | What the agent can do afterwards |
|---|---|---|
| 1 | An agent with no tools | Answer from what the model learned in training, and guess the rest |
| 2 | A first tool | Know the dates of this weekend |
| 3 | A tool that calls a web service | Know the real weather forecast |
| 4 | A knowledge base | Look up fares and hostel prices in our travel guide |
| 5 | A calculator and rules | Check the budget exactly and follow our rules |
| 6 | Memory | Answer a follow-up question |
| 7 | A linear workflow (Part 2) | Collect the facts with agent 1, write the plan with agent 2 |
| 8 | A workflow with a team of agents | Research both cities at the same time, one agent per city |

**A workflow or an agent?** Anthropic calls both *agentic systems* and draws one line between them ([Building effective agents](https://www.anthropic.com/research/building-effective-agents), December 2024):

- A **workflow** runs LLMs and tools along "predefined code paths": our code fixes the order of the steps.
- **Agents** are "systems where LLMs dynamically direct their own processes and tool usage": the model decides its next step.

Steps 1 to 6 build one **agent**: we never write the order of the tool calls. Step 7 builds the simplest **workflow**: two agents in a line, where our code fixes the order. Step 8 builds a workflow whose middle step runs in parallel: an orchestrator agent, one agent per city, and a synthesiser agent. Every agent in this workbook is built with `create_agent`.

After this workbook you can explain what a tool is and who runs it, read an agent's steps, give an agent knowledge and memory, and build a workflow as a graph, and say when a workflow, one agent or a team of agents is worth it.

> **Note.** The saved outputs come from a run on 4 November 2026 with Claude Haiku 5.5. Claude's answers change from run to run, and so does the weather.

## Setup: the key and the model

The agent uses Claude through the Claude Platform API. An **API key** is a secret that identifies you and is billed for every request. It never goes into a notebook. It goes into a file `.env` in the `workbooks` folder, with one line `ANTHROPIC_API_KEY=your key`. Copy `.env.example` to `.env` and paste your key; git ignores `.env`.

- `load_dotenv(find_dotenv(usecwd=True))` from `python-dotenv` finds the file, starting from the folder where the notebook runs, and loads the key into the environment.
- If no key is found, `getpass` asks for it without showing it on screen.

In [1]:
import os
from getpass import getpass

from dotenv import find_dotenv, load_dotenv

# Read ANTHROPIC_API_KEY from the file .env into the environment.
load_dotenv(find_dotenv(usecwd=True))
# No key found: ask for it; getpass hides what you type.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")
print("API key found. Its value is not printed.")

API key found. Its value is not printed.


`ChatAnthropic` from `langchain-anthropic` is the model object. We use **Claude Haiku 5.5** (`claude-haiku-5-5`), the smallest and cheapest Claude model, because an agent calls the model many times for one question. `max_tokens` is the longest reply allowed; always set it.

In [2]:
from langchain_anthropic import ChatAnthropic

# The model object: which Claude model, and the longest reply in tokens.
model = ChatAnthropic(model="claude-haiku-5-5", max_tokens=4096)

## A model call or an agent?

We already have `model`, and `model.invoke(...)` sends a question to Claude. So why build an agent? Both send our question to the same model. The difference is **what happens after Claude replies**.

`create_agent(model=model, tools=tools, system_prompt=...)` from `langchain.agents` builds an agent around the model. A **system prompt** is the developer's instruction for the whole conversation; a **user message** is what the user asks. We name both in full: `SYSTEM_PROMPT` and `USER_MESSAGE_GOAL`.

| | `model.invoke(messages)` | `agent.invoke({"messages": messages})` |
|---|---|---|
| Input | a text, or a list of `(role, text)` messages | a dictionary with the key `"messages"` |
| Output | one message (`AIMessage`); `.text` is the answer | a dictionary; `result["messages"]` is the whole conversation, and its last message is the answer |
| Model calls | exactly one | one or more, until the model stops asking for tools |
| Tools | the model can only *ask* for a tool; our own code must run it and call the model again | the agent runs the tool and calls the model again by itself |
| System prompt | sent as a `("system", ...)` message in every call | given once, to `create_agent` |

**A model call** is one request and one reply:

```mermaid
graph LR;
    user_message[User message] --> claude[Claude: one model call];
    claude --> reply[Reply: one AIMessage];
```

**An agent** is a loop around the model. After every reply it checks: did the model ask for a tool? If yes, it runs the tool, adds the result to the conversation and calls the model again. If no, the last reply is the answer.

```mermaid
graph LR;
    user_message[User message] --> model_node[Model call];
    model_node --> decision{Tool calls in the reply?};
    decision -- yes --> tools_node[Run the tools in our Python];
    tools_node -- results added to the messages --> model_node;
    decision -- no --> answer[Answer: the last message];
```

### Printing replies and results readably

A reply is an object with many fields, and printed as it is, it fills one long line. The helper `show_json` prints it as **indented JSON**, one field per line:

- `.model_dump()` turns a LangChain message (or any Pydantic object) into a dictionary.
- `json.dumps(value, indent=2)` writes a dictionary as JSON text, indented by two spaces. `default=` names a function for the values `json` cannot write by itself, such as a message inside a list. `ensure_ascii=False` keeps characters like € and ä as they are.
- `display(Markdown(text))` from `IPython.display` shows a text with headings, bold and tables, as Claude writes its answers.

In [3]:
import json

from IPython.display import Markdown, display


# json.dumps calls this for every value it cannot write by itself.
def to_dictionary(value):
    """Turn a message or a Pydantic object into a dictionary; anything else into text."""
    if hasattr(value, "model_dump"):
        return value.model_dump()
    return str(value)


def show_json(value) -> None:
    """Print a reply, a result or a tool result as indented JSON."""
    # indent=2: one field per line; ensure_ascii=False: keep € and ä.
    print(json.dumps(value, indent=2, ensure_ascii=False, default=to_dictionary))

### Model call and agent on the same question

The cell runs the same question both ways, with no tools, and prints what each call returns.

In [4]:
from langchain.agents import create_agent

SYSTEM_PROMPT = "You plan short, cheap trips for students who live in Berlin."
USER_MESSAGE_GOAL = "Plan a cheap trip from Berlin for this weekend: 2 people, 300 EUR in total. Prague or Vienna?"

# 1. The model alone: one call, one reply.
reply = model.invoke([("system", SYSTEM_PROMPT), ("user", USER_MESSAGE_GOAL)])
print("model.invoke returns:", type(reply).__name__)

# 2. The same model inside an agent, still with no tools.
agent = create_agent(model=model, tools=[], system_prompt=SYSTEM_PROMPT)
result = agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]})
print("agent.invoke returns:", type(result).__name__, "with the keys", list(result))
print("messages in result  :", [message.type for message in result["messages"]])

model.invoke returns: AIMessage


agent.invoke returns: dict with the keys ['messages']
messages in result  : ['human', 'ai']


**The whole reply.** `show_json(reply)` prints every field of the `AIMessage`. Look for `content` (the answer), `type` (`"ai"`), `tool_calls` (empty: no tools yet), `usage_metadata` (the tokens read and written) and `response_metadata` (the model name and why it stopped, `stop_reason`).

In [5]:
show_json(reply)

{
  "content": [
    {
      "signature": "CAQS8hIKEAgTGAI4AUIIdGhpbmtpbmcSDDiQy2hSz3F+1HnISxoM0MBF3hd0rvZSa2jtIjBqXmhpOaUooOVlB935ky5WPA85B77oZbntNJyIhYDq8YEwVzBGTMvioSAwS/jBkXMqjxKE8jLy2US9xxljkMrkaRLdxdZ9puRiURD1tnkf44DPeEE/mwipkHfNLmYh9imye4j7mRgZhxNatnVZEcoKX3HBAmSaa3KzbUhKoRc7JqN/t0nIWzEY2gXRTMFgIch/ZL2TbbIGq8JFNOD0CnY/8sCezv1qM05DlG9es7fhJD+FzeNOvN45y5qAIwLNQ5v9J0X25BYRndU1E4jUY0yCH5geHNOnZqXJOJxjGpm01UvgJvX67cL6Le4ssb6JJPZhouF4DLQfyNx2HZT/i6+WG+jXym5/USa5zozuziNqbLG7Q4HkWNVsk1acAF1oW187byzoEi56GXZ/2U6982WZHcc9cqbp6HVNi//gMBaQvY3dQ/DmSPCTlo2wSnkgnFobu2buDh48RM67ujpyLjpzjFJOl9uddC61SEMCd8vtJ3iO28r3iTyznRTZmoIG4DXE0vIOa64Dox9ZTmWysDOF7b5DxlbJlwOHYg1MJlYZNnRfg4S54qjDO5ZWCb5WK7nAwFx7417qz4verbdTY78U8O3duKBx/4XfE+JNlvyOHMbr/W6IU2nW77lZD84R13jxs3juRKB9/R8SbVyW5BwMyJP4qaWW+rHM9eCeE9gqNbWg5oXeVxn8ZVzPVPfB8oLvf5H5s+/1ZhdC4lObSPy3ldqoiwKxgUmZ6fhJiRiV6ohqOZNU3jl6/AR25S1EsvpHi7qoaCr77NLwpWEv3EPqp43iptMXq43A9H0Ixy1tFrmo91GoEPOt0lPCSeiQ7BFXYI8pDaMcgXyGX9oy8UL3/+u2R/btcna/7N029TKaOEFfwr3DnIETr

**The whole agent result.** `show_json(result)` prints the dictionary with the key `messages`: first our question (`"type": "human"`), then the reply (`"type": "ai"`), with the same fields as above.

In [6]:
show_json(result)

{
  "messages": [
    {
      "content": "Plan a cheap trip from Berlin for this weekend: 2 people, 300 EUR in total. Prague or Vienna?",
      "additional_kwargs": {},
      "response_metadata": {},
      "type": "human",
      "name": null,
      "id": "c849cf24-5cfc-4517-b778-81e8813b15e0"
    },
    {
      "content": [
        {
          "signature": "CAQSnBQKEAgTGAI4AUIIdGhpbmtpbmcSDHB28KM9mJpxXmqzGhoMKbOQFQrgXpEFSBRvIjCxNsbttneiYBmybT81EKd5xtybr3vvhPfc+Op9bN437CyaWrmLpK4M26XXtSsBsNoquRO7UsesM0cI8QSlwaX0b8QgeevCi8k3ZTgNhsBKo7Q0CEh9vqX3KHW53Ts/3OD0QmuKBKItXuA7Ec1t5GJqpj6EEu2SZiQU9gEC1m3JSa7NvSbWiA2uXUiAxNKRE6he3wPFCxFhsyfe3LJMYtKDj4sJFXr+btCT/SOFJSN2B0Tih7UFUAcelYoHIhk4yHJr4OYOa6M4sEG+Fx8/QhgkQHuppIotY6Nh7hxWLRkVZgAK0pqNVesp3L+uHpSmfj79TdFnCi62U6iYK4w8OG2sSX5w4at0uVDyFHxqzWrSXWyCAP3hV3p/oQR/yjSSN8/qqsRlU5VTrYiPFOrfbvk0Dp4Y77FW3ZEZu0H0pT//j0UD4nXHV4FQSAsf30YP7fI/Z8jU2IQkUhdKMUbp+I4hEnue6Avqpp+H43QKGHlKYjJ2F7I82fosVCo51FS47e9W5HSrICHJ6dcSzR1OnMEHrYnPD5M5+91wyLnCHCDTveR3HfX71M7N0FlT

`message.type` is `"human"` for a user message and `"ai"` for a model reply. With no tools, the agent made one model call, just like `model.invoke`: the conversation is the question and one reply. So without tools, an agent is only a model call in a different wrapper. The difference starts in Step 2, when the model can ask for a tool: `model.invoke` stops at the request, and the agent goes on.

## Step 1: an agent with no tools

The agent from the cell above has no tools, so it is only the model. Its last message is the answer; `.text` gives the text of a message, and `display(Markdown(...))` shows it with its headings and tables.

In [7]:
display(Markdown(result["messages"][-1].text))

**Prague.** Two nights for two people fits comfortably in 300 EUR. Vienna usually doesn't.

**Rough budget for 2 people, 2 nights (Fri–Sun), prices vary by booking time:**

| Item | Prague | Vienna |
|---|---|---|
| Return bus/train (Berlin) | 80–120 | 100–180 |
| Hostel, 2 nights (shared or private room) | 100–130 | 180–260 |
| Food (~15 EUR/day each) | 70–90 | 120–150 |
| Sights and extras | 20–30 | 30–50 |
| **Total** | **~270–370** | **~430–640** |

**Sample plan:**
- **Fri:** Take the evening Flixbus or EuroCity train from Berlin (about 4–5 hours). Book early for the cheapest fares. Check in, walk Old Town Square and the Charles Bridge at night, and have a cheap dinner and beer.
- **Sat:** Walk up to Prague Castle and Lesser Town, then visit the Jewish Quarter (a combined ticket saves money). Eat at a *lokál* for traditional food at low prices. Visit Letná Park for views in the afternoon.
- **Sun:** Pick up a cheap breakfast, then head back in the afternoon or evening.

**Money-saving tips:**
- Book transport at least 2–3 weeks ahead, or go with a late-night bus to save a hostel night.
- Pay in Czech koruna (CZK) and avoid exchange booths near the main squares. Use an ATM or a card without foreign fees.
- Choose a hostel outside the immediate center, or one with breakfast included.
- Free walking tours and free views from the bridges and parks cost nothing.

Prices change, so check current fares on Flixbus, DB, and hostel sites before you book. If you go Friday, book soon, since weekend fares rise as the date approaches.

**Discuss.** The answer may sound good. Now check it line by line. Which of these could the model really know?

- the dates of "this weekend"
- the weather on those days
- bus fares and hostel prices today
- whether the plan fits 300 EUR

None of them: a model knows only its training data, and it does not know today's date. It either says so or **guesses**. Each of the next steps fixes one of these gaps with a tool.

## Step 2: a first tool

### What a tool is

A **tool** is a Python function that the model may ask to run. The model never runs it. It writes a request with the tool's name and arguments; LangChain runs the function in *our* Python; the result goes back to the model as text.

The `@tool` decorator from `langchain.tools` turns a function into a tool. Three parts of it reach the model: the **name** (the function name), the **description** (the docstring) and the **arguments** (from the type hints). So the docstring is an instruction for the model: say when to use the tool.

Our first tool needs no internet: it computes the coming Saturday and Sunday from the computer's clock.

- `date.today()` is today's date; `.weekday()` gives 0 for Monday up to 6 for Sunday.
- `(5 - weekday) % 7` is the number of days until Saturday; `timedelta(days=n)` is a step of n days.
- The tool returns JSON text with `json.dumps`, which a model reads well. `json.loads` reads such a text back into a dictionary, so `show_json` can print it indented.

In [8]:
from datetime import date, timedelta

from langchain.tools import tool


# @tool turns the function into a tool: name, docstring and arguments go to the model.
@tool
def next_weekend() -> str:
    """Return today's date and the dates of the coming Saturday and Sunday. Call it when the user says "this weekend"."""
    today = date.today()
    # weekday() is 0 for Monday ... 6 for Sunday, so (5 - weekday) % 7 is the days until Saturday.
    saturday = today + timedelta(days=(5 - today.weekday()) % 7)
    sunday = saturday + timedelta(days=1)
    # Return JSON text: the model reads it well, and json.loads can read it back.
    return json.dumps({"today": today.isoformat(), "saturday": saturday.isoformat(), "sunday": sunday.isoformat()})

Test every tool by hand before a model gets it: `.invoke()` with a dictionary of arguments calls it the same way the agent will. A bug in a tool looks like a mistake of the model later.

In [9]:
show_json(json.loads(next_weekend.invoke({})))
print(next_weekend.name, "|", next_weekend.description)

{
  "today": "2026-11-04",
  "saturday": "2026-11-07",
  "sunday": "2026-11-08"
}
next_weekend | Return today's date and the dates of the coming Saturday and Sunday. Call it when the user says "this weekend".


### Reading the agent's steps

To see what the agent did, we print its conversation. A model message (`type == "ai"`) can hold **tool calls** in `.tool_calls`: each is a dictionary with the tool's `name` and `args`. A tool message (`type == "tool"`) holds the result as text in `.content`, and the tool's name in `.name`. The helper `show_run` prints each request with its arguments, each result, and then the answer as Markdown.

- Tool results that are JSON text are read with `json.loads` and printed indented; plain text, such as a note from the travel guide, is printed as it is.
- `try: ... except json.JSONDecodeError:` catches the error that `json.loads` raises for text that is not JSON.

In [10]:
def show_tool_result(content: str) -> None:
    """Print a tool result: indented if it is JSON, as it is otherwise."""
    try:
        # JSON text: print it indented.
        show_json(json.loads(content))
    except json.JSONDecodeError:
        # Plain text, such as a note from the travel guide: print it as it is.
        print(content)


def show_run(result: dict) -> None:
    """Print every tool request of the model, every tool result, and the final answer."""
    # Walk through the whole conversation, from the question to the answer.
    for message in result["messages"]:
        # A model reply: print each tool it asks for, with the arguments.
        if message.type == "ai":
            for call in message.tool_calls:
                print(f"model asks for: {call['name']}")
                show_json(call["args"])
                print()
        # A tool message: print the result that our Python returned.
        elif message.type == "tool":
            print(f"tool returns: {message.name}")
            show_tool_result(message.content)
            print()
    # The last message is the answer; show it as Markdown.
    display(Markdown("**ANSWER**\n\n" + result["messages"][-1].text))

**Predict.** The agent now has the tool `next_weekend`. Will it call it before it answers?

In [11]:
agent = create_agent(model=model, tools=[next_weekend], system_prompt=SYSTEM_PROMPT)
show_run(agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]}))

model asks for: next_weekend
{}

tool returns: next_weekend
{
  "today": "2026-11-04",
  "saturday": "2026-11-07",
  "sunday": "2026-11-08"
}



**ANSWER**

**Prague.** It's the better fit for 300 EUR for two people.

I can't check live prices, so these are rough estimates for one night (Sat–Sun) in early November. Verify them before booking.

| | Prague | Vienna |
|---|---|---|
| Travel, round trip for 2 | ~80–140 EUR (bus or RegioJet) | ~120–200 EUR (farther, pricier trains) |
| Double room, 1 night | ~50–80 EUR | ~80–120 EUR |
| Food and activities | ~60–80 EUR | ~80–100 EUR |
| **Estimated total** | **~200–300 EUR** | **~280–400 EUR** |

Prague is about 350 km from Berlin, roughly 4–5 hours by bus or train. Vienna is about 680 km away, so it takes 6–8 hours, and you'd likely pay more for a shorter stay there. Vienna would only fit the budget if you found unusually cheap tickets and a hostel.

To keep the cost down:
- Compare FlixBus, RegioJet, and Deutsche Bahn Sparpreis fares for Sat morning out and Sun evening back.
- Book a hostel or budget private room near a metro line, not in the Old Town center.
- Eat at local canteens and pubs rather than tourist restaurants, and stick to the free sights like Charles Bridge, the Old Town Square, and Vyšehrad.

Check your ID before you go. Germany has had temporary checks on some internal borders, so confirm the current situation for the Czech border.

The model chose to call the tool, read the dates, and then answered. We did not tell it to: the docstring said when the tool is useful. This loop is what makes an **agent**: the model reads, asks for a tool, reads the result, and decides again, until it can answer. The dates are now right, but the weather and the prices are still guesses.

Who does what in this run: Claude only writes text, including the tool request; our Python runs the tool.

```mermaid
sequenceDiagram
    participant User
    participant Agent as Agent loop in our Python
    participant Claude
    participant Tool as next_weekend in our Python
    User->>Agent: USER_MESSAGE_GOAL
    Agent->>Claude: model call 1: question and tool descriptions
    Claude-->>Agent: tool request next_weekend
    Agent->>Tool: run next_weekend
    Tool-->>Agent: dates as JSON
    Agent->>Claude: model call 2: question, request and dates
    Claude-->>Agent: answer, no tool request
    Agent-->>User: last message
```

### The model alone, with the same tool

Now the comparison from the start of the workbook, with a tool. `model.bind_tools(tools)` gives the model the tool descriptions, without an agent around it. **Predict.** What does `model.invoke` return now?

In [12]:
# Give the model the tool description, but no agent loop around it.
model_with_tools = model.bind_tools([next_weekend])
reply = model_with_tools.invoke([("system", SYSTEM_PROMPT), ("user", USER_MESSAGE_GOAL)])
# The reply holds the tool request; nobody runs the tool.
print("tool calls:")
show_json(reply.tool_calls)
print("text      :", reply.text or "(empty: no answer yet)")

tool calls:
[
  {
    "name": "next_weekend",
    "args": {},
    "id": "toolu_01Tj9mJSSCsZht1GD5ByNk8p",
    "type": "tool_call"
  }
]
text      : (empty: no answer yet)


The model alone stops after model call 1: it returns the tool request, nobody runs the tool, and we get no answer. To finish, our own code would have to run `next_weekend`, add the result to the messages and call the model again, and repeat this for every tool request. This loop is exactly what `create_agent` writes for us.

## Step 3: a tool that calls a web service

The weather changes every day, so it must come from a web service. **Open-Meteo** gives forecasts for free, without a key, for non-commercial use such as teaching. The data is under the CC BY 4.0 licence, so the tool returns the credit "Weather data by Open-Meteo.com".

The tool makes two requests:

1. the **geocoding** service turns the city name into latitude and longitude;
2. the **forecast** service gives, per day, the lowest and highest temperature and the highest chance of rain.

`httpx.get(url, params=..., timeout=10)` sends a request; `params` becomes the query string, and `timeout` stops waiting after 10 seconds. `.json()` reads the answer into a dictionary. `zip(a, b, c)` walks through several lists side by side.

If something goes wrong, the tool returns a sentence that starts with "Error", instead of crashing. The model reads this sentence like any other result and can react to it.

In [13]:
import httpx

GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"


# parse_docstring=True: each line under Args: describes one argument to the model.
@tool(parse_docstring=True)
def weather_forecast(city: str, start_date: str, end_date: str) -> str:
    """Get the daily weather forecast for a city. Works only for dates up to 16 days from today.

    Args:
        city: City name in English, for example "Prague" (not "Praha").
        start_date: First day, as YYYY-MM-DD.
        end_date: Last day, as YYYY-MM-DD.
    """
    try:
        # Request 1, geocoding: the city name becomes latitude and longitude.
        places = httpx.get(GEOCODING_URL, params={"name": city, "count": 1, "language": "en"}, timeout=10).json()
        # No match: return an error sentence that the model can read and react to.
        if not places.get("results"):
            return f"Error: no city called {city!r} was found. Try the English name."
        # Take the best match.
        place = places["results"][0]
        # Request 2, forecast: per day the lowest and highest temperature and the highest chance of rain.
        response = httpx.get(FORECAST_URL, timeout=10, params={
            "latitude": place["latitude"], "longitude": place["longitude"], "timezone": "auto",
            "start_date": start_date, "end_date": end_date,
            "daily": "temperature_2m_min,temperature_2m_max,precipitation_probability_max",
        })
        # An error status (for example a date too far ahead) becomes an exception.
        response.raise_for_status()
    # Any network or status error: an error sentence instead of a crash.
    except httpx.HTTPError as error:
        return f"Error: the weather service did not answer ({type(error).__name__}). It forecasts at most 16 days ahead."
    # The forecast holds one list per measure, all in the same day order.
    daily = response.json()["daily"]
    # zip walks through the four lists side by side and builds one dictionary per day.
    days = [
        {"date": day, "min_c": low, "max_c": high, "rain_chance_pct": rain}
        for day, low, high, rain in zip(daily["time"], daily["temperature_2m_min"],
                                        daily["temperature_2m_max"], daily["precipitation_probability_max"])
    ]
    # The licence CC BY 4.0 asks for the credit, so it travels with the data.
    return json.dumps({"city": place["name"], "country": place["country"], "days": days,
                       "source": "Weather data by Open-Meteo.com"})

`parse_docstring=True` reads the `Args:` section, so each argument gets its own description for the model. `raise_for_status()` turns an answer with an error status into an exception, which the `except` catches. Test the tool by hand for this weekend:

In [14]:
# Test the tool by hand with this weekend's dates, as the agent will call it.
weekend = json.loads(next_weekend.invoke({}))
show_json(json.loads(weather_forecast.invoke({"city": "Prague", "start_date": weekend["saturday"], "end_date": weekend["sunday"]})))

{
  "city": "Prague",
  "country": "Czechia",
  "days": [
    {
      "date": "2026-11-07",
      "min_c": 2.1,
      "max_c": 6.1,
      "rain_chance_pct": 4
    },
    {
      "date": "2026-11-08",
      "min_c": 3.1,
      "max_c": 7.2,
      "rain_chance_pct": 13
    }
  ],
  "source": "Weather data by Open-Meteo.com"
}


**Predict.** The weather tool needs dates, but the user did not give any. What will the agent do first?

In [15]:
agent = create_agent(model=model, tools=[next_weekend, weather_forecast], system_prompt=SYSTEM_PROMPT)
show_run(agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]}))

model asks for: next_weekend
{}

tool returns: next_weekend
{
  "today": "2026-11-04",
  "saturday": "2026-11-07",
  "sunday": "2026-11-08"
}

model asks for: weather_forecast
{
  "city": "Prague",
  "start_date": "2026-11-07",
  "end_date": "2026-11-08"
}

model asks for: weather_forecast
{
  "city": "Vienna",
  "start_date": "2026-11-07",
  "end_date": "2026-11-08"
}

tool returns: weather_forecast
{
  "city": "Prague",
  "country": "Czechia",
  "days": [
    {
      "date": "2026-11-07",
      "min_c": 2.1,
      "max_c": 6.1,
      "rain_chance_pct": 4
    },
    {
      "date": "2026-11-08",
      "min_c": 3.1,
      "max_c": 7.2,
      "rain_chance_pct": 13
    }
  ],
  "source": "Weather data by Open-Meteo.com"
}

tool returns: weather_forecast
{
  "city": "Vienna",
  "country": "Austria",
  "days": [
    {
      "date": "2026-11-07",
      "min_c": 2.0,
      "max_c": 9.3,
      "rain_chance_pct": 3
    },
    {
      "date": "2026-11-08",
      "min_c": 0.8,
      "max_c": 7.8

**ANSWER**

**Prague is the better pick for 300 EUR for two.**

**Weather (Sat 7 – Sun 8 Nov):** Both cities are cold and mostly dry. Prague is 2–7 °C with 4–13% rain chance. Vienna is 1–9 °C with 3–5% rain chance, and it will be a bit colder Sunday morning. Weather doesn't separate them much.

**Budget:** I can't check live fares or hotel prices, so these are rough estimates to verify before booking:
- **Prague:** A FlixBus or train from Berlin is roughly 20–50 EUR each way, and the trip takes about 4–5 hours. Hostel beds are often 20–35 EUR a night, and beer and food are cheap. A two-night trip for two should come in around 200–250 EUR, leaving some slack.
- **Vienna:** The trip is longer, about 7–8 hours by train or bus, and transport is usually 30–60 EUR each way. Accommodation and food cost more. Two nights for two will be tight against 300 EUR and may need a night bus to save a night's hotel.

**Recommendation:** Go to Prague. It's cheaper, closer, and easier to fit in a weekend. Book a bus early, since weekend fares rise. Pick Vienna only if you specifically want its museums and cafés and are willing to spend closer to the limit.

Read the `model asks for` lines. The model first asked for the dates, then passed them to the weather tool: the result of one tool became the input of the next. It may also ask for both cities in one step (**parallel tool calls**), because the two forecasts do not depend on each other. The weather is now real. The fares and hostel prices are still guesses.

## Step 4: a knowledge base

### Our own travel guide

Fares and hostel prices are not in any free web service. Many real agents have the same problem: the facts they need are in the company's own documents. The answer is a **knowledge base**: our own texts, and a tool that searches them. This is the retrieval step of RAG from notebook 01 of this session, used as a tool: the model decides *when* to look something up.

Our knowledge base is a short travel guide, one note per city. **The guide is written for this course: the prices are rough example values, not real offers.** Check real prices before you book.

In [16]:
# Our knowledge base: one note per city, with example prices written for the course.
TRIP_GUIDE = [
    "Prague, Czechia. Currency: Czech koruna (CZK). Bus from Berlin: about 4.5 hours, return fare about 40 EUR per person. "
    "Hostel bed: about 25 EUR per night. Cheap food: a lunch menu costs about 8 EUR. "
    "Free: Charles Bridge early in the morning, and the view from Letna Park.",
    "Vienna, Austria. Currency: euro (EUR). Bus from Berlin: about 9.5 hours, return fare about 70 EUR per person. "
    "Hostel bed: about 35 EUR per night. Cheap food: a sausage stand costs about 5 EUR. "
    "Free: the flea market at the Naschmarkt on Saturday morning.",
    "Dresden, Germany. Currency: euro (EUR). Bus from Berlin: about 2.5 hours, return fare about 25 EUR per person. "
    "Hostel bed: about 25 EUR per night. Free: a walk along the Elbe meadows, and the Neustadt district.",
]

### A simple search tool

Notebook 01 searches with embeddings. For three notes, a **keyword search** is enough and easy to follow:

- `words(text)` turns a text into a set of lower-case words; `re.findall(r"[a-z]+", ...)` finds all runs of letters.
- `query_words & words(note)` is the set of words that the query and the note share. The note with the most shared words ranks first.
- `sorted(..., key=..., reverse=True)` sorts from the highest score down. The tool returns the best two notes that share at least one word.

In [17]:
import re


def words(text: str) -> set[str]:
    """The set of lower-case words in a text."""
    # [a-z]+ finds every run of letters; the set removes repeated words.
    return set(re.findall(r"[a-z]+", text.lower()))


@tool
def search_trip_guide(query: str) -> str:
    """Search our student travel guide for bus fares, travel times, hostel prices and free things to do. Put the city name in the query."""
    # Score each note by the number of words it shares with the query (& is set intersection).
    query_words = words(query)
    ranked = sorted(TRIP_GUIDE, key=lambda note: len(query_words & words(note)), reverse=True)
    # Keep the best two notes, but only those that share at least one word.
    best_notes = [note for note in ranked[:2] if query_words & words(note)]
    if not best_notes:
        return "No note matches. The guide covers Prague, Vienna and Dresden."
    return "\n\n".join(best_notes)


# Test the tool by hand.
print(search_trip_guide.invoke({"query": "hostel price Prague"}))

Prague, Czechia. Currency: Czech koruna (CZK). Bus from Berlin: about 4.5 hours, return fare about 40 EUR per person. Hostel bed: about 25 EUR per night. Cheap food: a lunch menu costs about 8 EUR. Free: Charles Bridge early in the morning, and the view from Letna Park.

Vienna, Austria. Currency: euro (EUR). Bus from Berlin: about 9.5 hours, return fare about 70 EUR per person. Hostel bed: about 35 EUR per night. Cheap food: a sausage stand costs about 5 EUR. Free: the flea market at the Naschmarkt on Saturday morning.


**Predict.** The model now has three tools. For the same question, which tools will it call, and how often?

In [18]:
agent = create_agent(model=model, tools=[next_weekend, weather_forecast, search_trip_guide], system_prompt=SYSTEM_PROMPT)
show_run(agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]}))

model asks for: next_weekend
{}

model asks for: search_trip_guide
{
  "query": "Prague"
}

model asks for: search_trip_guide
{
  "query": "Vienna"
}

tool returns: next_weekend
{
  "today": "2026-11-04",
  "saturday": "2026-11-07",
  "sunday": "2026-11-08"
}

tool returns: search_trip_guide
Prague, Czechia. Currency: Czech koruna (CZK). Bus from Berlin: about 4.5 hours, return fare about 40 EUR per person. Hostel bed: about 25 EUR per night. Cheap food: a lunch menu costs about 8 EUR. Free: Charles Bridge early in the morning, and the view from Letna Park.

tool returns: search_trip_guide
Vienna, Austria. Currency: euro (EUR). Bus from Berlin: about 9.5 hours, return fare about 70 EUR per person. Hostel bed: about 35 EUR per night. Cheap food: a sausage stand costs about 5 EUR. Free: the flea market at the Naschmarkt on Saturday morning.

model asks for: weather_forecast
{
  "city": "Prague",
  "start_date": "2026-11-07",
  "end_date": "2026-11-08"
}

model asks for: weather_forecast


**ANSWER**

**Prague is the better choice.** Both cities fit the 300 EUR budget for one night, but Prague leaves much more room and the trip is far shorter.

**Dates:** Saturday 7 November to Sunday 8 November 2026

**Estimated cost for 2 people (one hostel night):**

| | Prague | Vienna |
|---|---|---|
| Bus from Berlin (return, 2 people) | 80 EUR | 140 EUR |
| Hostel (1 night, 2 beds) | 50 EUR | 70 EUR |
| **Subtotal** | **130 EUR** | **210 EUR** |
| Left for food and extras | ~170 EUR | ~90 EUR |
| Bus time each way | ~4.5 h | ~9.5 h |

If you want two nights, Prague costs about 180 EUR, leaving about 120 EUR. Vienna costs about 280 EUR, leaving about 20 EUR, which is too tight.

**Weather (Nov 7–8):**
- Prague: 2–7 °C, 4–13% chance of rain
- Vienna: 1–9 °C, 3–5% chance of rain

Both are cold but dry, so pack warm layers either way.

**Things to do on a budget:**
- **Prague:** See Charles Bridge early in the morning before the crowds, go up to Letna Park for the view (both free), and eat the lunch menu, which costs about 8 EUR.
- **Vienna:** Visit the Naschmarkt flea market on Saturday morning (free) and eat a sausage from a stand for about 5 EUR.

Prague gives you a cheaper, shorter trip with more money left over. Vienna is worth it only if you can spare the extra cost and about 19 hours of bus travel over the weekend.

The prices in the answer now come from our guide, not from the model's memory. Check one number from the answer against the `tool returns` lines. One gap is left: the agent adds up fares and beds in its head, and nothing stops it from recommending a plan over the budget.

## Step 5: a calculator and rules

### A budget tool

A language model can make arithmetic mistakes, and it can give a different sum each time. A sum that must be right belongs in Python. The tool `budget_per_person` takes the fare and the bed price from the guide and returns what is left per person for food and activities, and whether the plan fits.

In [19]:
@tool(parse_docstring=True)
def budget_per_person(total_budget_eur: float, people: int, nights: int,
                      return_fare_eur: float, bed_per_night_eur: float) -> str:
    """Compute what is left per person for food and activities after the fare and the beds.

    Args:
        total_budget_eur: Budget of the whole group, in euros.
        people: Number of travellers.
        nights: Number of nights away.
        return_fare_eur: Return fare for one person, in euros.
        bed_per_night_eur: Price of one bed for one night, in euros.
    """
    # The sums happen in Python, so they are exact and the same every time.
    per_person = total_budget_eur / people
    left = per_person - return_fare_eur - nights * bed_per_night_eur
    return json.dumps({"budget_per_person_eur": round(per_person, 2),
                       "left_for_food_and_activities_eur": round(left, 2), "within_budget": left >= 0})


# Test by hand: 300 EUR, 2 people, 1 night, Prague's fare and bed price.
show_json(json.loads(budget_per_person.invoke({"total_budget_eur": 300, "people": 2, "nights": 1,
                                               "return_fare_eur": 40, "bed_per_night_eur": 25})))

{
  "budget_per_person_eur": 150.0,
  "left_for_food_and_activities_eur": 85.0,
  "within_budget": true
}


150 EUR per person, minus 40 EUR for the bus and 25 EUR for one night, leaves 85 EUR for Prague.

### Rules in the system prompt

The system prompt is where the **rules** for the agent go. Good rules are concrete and can be checked: "be careful with money" is not; "never recommend a plan over the budget" is.

In [20]:
# All four tools, used by every agent from now on.
TOOLS = [next_weekend, weather_forecast, search_trip_guide, budget_per_person]
# Concrete rules that can be checked; they go into the system prompt.
RULES = """You plan short, cheap trips for students who live in Berlin.
Rules:
- Use the tools for every fact: dates, weather, fares, prices and sums. Never guess a number.
- Prices from the travel guide are course examples. Say so in your answer.
- Never recommend a plan over the budget. If nothing fits, say so.
- Answer in at most 10 lines: the city and why, the dates, the travel, the money left per person, the weather and what to pack."""

# The agent of Part 1: the model, the four tools and the rules.
planner = create_agent(model=model, tools=TOOLS, system_prompt=RULES)
result = planner.invoke({"messages": [("user", USER_MESSAGE_GOAL)]})
show_run(result)

model asks for: next_weekend
{}

model asks for: search_trip_guide
{
  "query": "Prague bus fare from Berlin, travel time, hostel prices, free things to do"
}

model asks for: search_trip_guide
{
  "query": "Vienna bus fare from Berlin, travel time, hostel prices, free things to do"
}

tool returns: next_weekend
{
  "today": "2026-11-04",
  "saturday": "2026-11-07",
  "sunday": "2026-11-08"
}

tool returns: search_trip_guide
Prague, Czechia. Currency: Czech koruna (CZK). Bus from Berlin: about 4.5 hours, return fare about 40 EUR per person. Hostel bed: about 25 EUR per night. Cheap food: a lunch menu costs about 8 EUR. Free: Charles Bridge early in the morning, and the view from Letna Park.

Vienna, Austria. Currency: euro (EUR). Bus from Berlin: about 9.5 hours, return fare about 70 EUR per person. Hostel bed: about 35 EUR per night. Cheap food: a sausage stand costs about 5 EUR. Free: the flea market at the Naschmarkt on Saturday morning.

tool returns: search_trip_guide
Vienna, Aust

**ANSWER**

**Recommendation: Prague.** It's cheaper and quicker, and both cities fit your 300 EUR budget.

- **Dates:** Sat 7 – Sun 8 November 2026, one night.
- **Travel:** Bus from Berlin takes about 4.5 hours in Prague vs 9.5 in Vienna. Return fare is about 40 EUR per person in Prague vs 70 in Vienna (guide examples, not live prices).
- **Lodging:** A hostel bed is about 25 EUR a night in Prague vs 35 in Vienna (guide examples).
- **Money left per person for food and activities:** 85 EUR in Prague vs 45 EUR in Vienna, after fare and bed for two people.
- **Weather in Prague:** 2–6 °C on Saturday and 3–7 °C on Sunday, with 4–13% chance of rain.
- **Weather in Vienna:** 2–9 °C on Saturday and 1–8 °C on Sunday, with 3–5% chance of rain.
- **What to pack:** Warm layers, a warm hat and gloves, and a light waterproof jacket for Prague.
- **Free things in Prague:** Charles Bridge early in the morning and the view from Letna Park.
- **Cheap food in Prague:** Lunch menus around 8 EUR (guide example).

### What the better answer cost

Every model reply carries `usage_metadata`: the tokens the model read (`input_tokens`) and wrote (`output_tokens`). The helper adds them up for one run.

In [21]:
def count_usage(result: dict) -> dict:
    """Count model calls, tool calls and tokens of one agent run."""
    # Only the model replies (type "ai") carry token counts.
    replies = [message for message in result["messages"] if message.type == "ai"]
    return {
        "model_calls": len(replies),
        # One reply can ask for several tools at once.
        "tool_calls": sum(len(reply.tool_calls) for reply in replies),
        "input_tokens": sum(reply.usage_metadata["input_tokens"] for reply in replies),
        "output_tokens": sum(reply.usage_metadata["output_tokens"] for reply in replies),
    }


show_json(count_usage(result))

{
  "model_calls": 3,
  "tool_calls": 7,
  "input_tokens": 6117,
  "output_tokens": 1436
}


Each model call sends the whole conversation again, with all tool descriptions. So the input grows with every step, and an agent with many steps reads its first question many times. At the Haiku 5.5 price on the Anthropic pricing page (3 November 2026: 0.10 USD per million input tokens, 0.50 USD per million output tokens), a run like this costs a fraction of a cent. Compare Step 1: one model call, no tools, and an answer full of guesses.

A system prompt is a request, not a guarantee: the model may still break a rule. A rule that must always hold goes into code, for example into a tool.

## Step 6: memory for a follow-up question

**Predict.** We ask a follow-up: "A third friend joins, so we are 3 people with 400 EUR. Does the plan still work?" What does `planner` answer?

In [22]:
USER_MESSAGE_FOLLOW_UP = "A third friend joins, so we are 3 people with 400 EUR. Does the plan still work?"
# A new invoke with no memory: the planner sees only this one message.
display(Markdown(planner.invoke({"messages": [("user", USER_MESSAGE_FOLLOW_UP)]})["messages"][-1].text))

I don't have the earlier plan, so I can't check it yet. To recalculate with 3 people and 400 EUR, I need:

1. **City** you're planning to visit.
2. **Dates or number of nights.**
3. **Return fare per person** (or let me look it up in the trip guide).
4. **Bed price per night** (or let me look that up too).

Once I have these, I'll run the budget for 3 people and check the weather for your dates. Note that guide prices are course examples, not live fares.

The planner does not know which plan we mean: every `invoke` starts an empty conversation. **Short-term memory** keeps the messages of one conversation between calls.

- A **checkpointer** saves the conversation after every step. `InMemorySaver` from `langgraph.checkpoint.memory` keeps it in the notebook's memory.
- A **thread** is one conversation, named by a `thread_id` in the config: `{"configurable": {"thread_id": "trip-1"}}`. The same `thread_id` continues it; a new one starts fresh.

```mermaid
graph LR;
    call_1[invoke 1: USER_MESSAGE_GOAL, thread trip-1] --> agent_node[memory_planner];
    agent_node -- saves the messages --> saver[(InMemorySaver: thread trip-1)];
    call_2[invoke 2: USER_MESSAGE_FOLLOW_UP, thread trip-1] --> agent_node;
    saver -- loads the earlier messages --> agent_node;
    agent_node --> answer[Answer to the follow-up];
```

In [23]:
from langgraph.checkpoint.memory import InMemorySaver

# The same planner, plus a checkpointer that saves the conversation after every step.
memory_planner = create_agent(model=model, tools=TOOLS, system_prompt=RULES, checkpointer=InMemorySaver())
# The thread_id names the conversation; the same id continues it.
thread = {"configurable": {"thread_id": "trip-1"}}

# Call 1: plan the trip; the messages are saved under trip-1.
memory_planner.invoke({"messages": [("user", USER_MESSAGE_GOAL)]}, thread)
# Call 2, same thread: the agent reads call 1 again before it answers.
follow_up_result = memory_planner.invoke({"messages": [("user", USER_MESSAGE_FOLLOW_UP)]}, thread)
display(Markdown(follow_up_result["messages"][-1].text))

Yes, Prague still works with 3 people and 400 EUR.

- **Money:** 400 EUR split three ways is 133.33 EUR per person for the fare and bed (40 EUR bus return plus 25 EUR hostel bed, both course examples from the guide).
- **Left for food and activities:** 68.33 EUR per person, down from 85 EUR with two people. Within budget.
- **Dates, travel and weather:** Unchanged from before: Sat 7 to Sun 8 November, about 4.5 hours by bus, 2–7 °C with low rain chance, so pack warm layers and a light waterproof jacket.

Now the agent remembers the plan and usually calls only `budget_per_person` again: the dates, the weather and the prices are already in the conversation. Memory saves tool calls, but every new call reads the whole saved conversation as input.

This is the end of Part 1: one agent with four tools, a knowledge base, rules and memory. Compare its answer with the one from Step 1.

## Step 7 (Part 2): a linear workflow

### Two agents in a line

From now on every model call goes through `create_agent`. Steps 7 and 8 do not add new syntax for the model; they connect several agents.

The planner of Step 5 does two jobs in one conversation: it collects facts with the tools, and it writes the plan. A **linear workflow** gives each job to its own agent and runs them one after the other:

1. `research_agent` has our four tools. It collects the facts for every city and writes them as a list, without a recommendation.
2. `writer_agent` has **no tools**. It reads the request and the facts, and writes the plan.

Our code passes the last message of agent 1 to agent 2. Anthropic calls this pattern **prompt chaining**: it "decomposes a task into a sequence of steps, where each LLM call processes the output of the previous one". Each agent gets an easier task, and between the two steps our code can read and check the facts.

```mermaid
graph LR;
    request[USER_MESSAGE_GOAL] --> research_agent[Agent 1: research_agent, with tools];
    research_agent -- facts --> writer_agent[Agent 2: writer_agent, no tools];
    writer_agent --> answer[Answer];
```

In [24]:
RESEARCH_RULES = """You collect facts for short, cheap trips from Berlin.
Rules:
- Use the tools for every fact: dates, weather, fares, prices and sums. Never guess a number.
- For every city in the request, list: the dates, the weather, the return fare, the bed price and the money left per person.
- Write only the facts, as a short list per city. Do not recommend a city."""

WRITER_RULES = """You plan short, cheap trips for students who live in Berlin.
Rules:
- Use only the facts you are given. You have no tools.
- Prices from the travel guide are course examples. Say so in your answer.
- Never recommend a plan over the budget. If nothing fits, say so.
- Answer in at most 10 lines: the city and why, the dates, the travel, the money left per person, the weather and what to pack."""

# Agent 1 gets the tools; agent 2 gets none, so it can only use the facts we pass on.
research_agent = create_agent(model=model, tools=TOOLS, system_prompt=RESEARCH_RULES)
writer_agent = create_agent(model=model, tools=[], system_prompt=WRITER_RULES)

**Predict.** Agent 2 has no tools. Where will the numbers in its answer come from?

The chain is two `invoke` calls: the text of agent 1's last message becomes part of agent 2's user message.

In [25]:
# Agent 1: collect the facts with the tools.
research_result = research_agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]})
# The text of agent 1's last message: the list of facts.
facts = research_result["messages"][-1].text
show_run(research_result)

# Agent 2: the request and the facts, in one user message.
writer_result = writer_agent.invoke({"messages": [("user", f"{USER_MESSAGE_GOAL}\n\nFacts:\n{facts}")]})
display(Markdown("**PLAN OF AGENT 2**\n\n" + writer_result["messages"][-1].text))

model asks for: next_weekend
{}

tool returns: next_weekend
{
  "today": "2026-11-04",
  "saturday": "2026-11-07",
  "sunday": "2026-11-08"
}

model asks for: weather_forecast
{
  "city": "Prague",
  "start_date": "2026-11-07",
  "end_date": "2026-11-08"
}

model asks for: weather_forecast
{
  "city": "Vienna",
  "start_date": "2026-11-07",
  "end_date": "2026-11-08"
}

model asks for: search_trip_guide
{
  "query": "Berlin to Prague bus fare return travel time"
}

model asks for: search_trip_guide
{
  "query": "Berlin to Vienna bus fare return travel time"
}

model asks for: search_trip_guide
{
  "query": "Prague hostel bed price"
}

model asks for: search_trip_guide
{
  "query": "Vienna hostel bed price"
}

tool returns: weather_forecast
{
  "city": "Prague",
  "country": "Czechia",
  "days": [
    {
      "date": "2026-11-07",
      "min_c": 2.1,
      "max_c": 6.1,
      "rain_chance_pct": 4
    },
    {
      "date": "2026-11-08",
      "min_c": 3.1,
      "max_c": 7.2,
      "rai

**ANSWER**

Dates: Saturday 7 November to Sunday 8 November 2026, which is 1 night. Both cities are within the 300 EUR budget for 2 people.

**Prague**
- Dates: 7–8 Nov 2026 (1 night)
- Weather: Sat 2.1–6.1 °C, 4% rain; Sun 3.1–7.2 °C, 13% rain
- Return bus fare: 40 EUR per person (about 4.5 hours each way)
- Hostel bed: 25 EUR per night
- Money left per person for food and activities: 85 EUR

**Vienna**
- Dates: 7–8 Nov 2026 (1 night)
- Weather: Sat 2.0–9.3 °C, 3% rain; Sun 0.8–7.8 °C, 5% rain
- Return bus fare: 70 EUR per person (about 9.5 hours each way)
- Hostel bed: 35 EUR per night
- Money left per person for food and activities: 45 EUR

**PLAN OF AGENT 2**

**Recommendation: Prague.** Both trips fit the 300 EUR budget, but Prague leaves 85 EUR per person against 45 EUR for Vienna, and the bus takes about 4.5 hours each way instead of 9.5.
**Dates:** Sat 7 to Sun 8 November 2026 (1 night).
**Travel:** Return bus 40 EUR per person (about 4.5 hours each way). Hostel bed 25 EUR per night.
**Money left per person:** 85 EUR for food and activities (150 EUR share minus 40 EUR bus and 25 EUR bed).
**Weather:** Sat 2.1–6.1 °C, 4% rain; Sun 3.1–7.2 °C, 13% rain.
**Pack:** warm coat, hat, gloves, layers, and a light waterproof for Sunday.
*(Vienna, for comparison: 70 EUR bus, 35 EUR bed, 45 EUR left, 0.8–9.3 °C, 3–5% rain.)*
*Note: the prices above are course examples from the travel guide, not live fares. Check current prices before booking.*

Every number in the plan of agent 2 must come from the facts of agent 1: agent 2 cannot call a tool. Check one number from the plan against the `tool returns` lines of agent 1.

### The same chain as a graph

Step 8 runs agents at the same time, and plain Python calls one after the other are no longer enough. So we write the chain again with **LangGraph**, the library that `create_agent` runs on. A LangGraph workflow is a **graph**:

- The **state** is the data that every node reads and updates. `ChainState` is a Pydantic class, as `TripPlan` in Step 8 will be: one field per value. A node reads a field with a dot, `state.request`.
- **In a dictionary, out a dictionary.** `trip_chain.invoke({"request": ...})` takes the input fields as a dictionary, not a list of messages as an agent does, and it returns the final state as a dictionary: `chain_result["answer"]`.
- A **node** is a Python function: it receives the state and returns a dictionary with the keys it changes. Each of our nodes runs one agent.
- An **edge** says which node runs next. `add_edge(a, b)` runs `b` after `a`; `START` and `END` are the entry and the exit.
- `compile()` makes the graph runnable; the result has `.invoke`, like an agent.

In [26]:
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel


# The state, as a Pydantic class: the fields that the nodes read and write.
class ChainState(BaseModel):
    # Set by us when we call invoke; no default, so the input must have it.
    request: str
    # Written by the node research; empty until then.
    facts: str = ""
    # Written by the node write; empty until then.
    answer: str = ""


# A node receives the state as an object (state.request) and returns a dict with only the keys it changes.
def research(state: ChainState) -> dict:
    """Agent 1: collect the facts with the tools."""
    result = research_agent.invoke({"messages": [("user", state.request)]})
    return {"facts": result["messages"][-1].text}


def write(state: ChainState) -> dict:
    """Agent 2: write the plan from the facts only."""
    result = writer_agent.invoke({"messages": [("user", f"{state.request}\n\nFacts:\n{state.facts}")]})
    return {"answer": result["messages"][-1].text}


# Build the graph: two nodes, and edges that fix the order START -> research -> write -> END.
chain_builder = StateGraph(ChainState)
chain_builder.add_node("research", research)
chain_builder.add_node("write", write)
chain_builder.add_edge(START, "research")
chain_builder.add_edge("research", "write")
chain_builder.add_edge("write", END)
# compile() makes the graph runnable with .invoke.
trip_chain = chain_builder.compile()

The edges are the order of the steps. They are written in our code, so the model cannot skip a step or add one. `time.perf_counter()` is a clock in seconds, so the difference of two readings is a duration; we keep the time for Step 8.

In [27]:
import time

# Read the clock before and after the run; the difference is the duration in seconds.
start = time.perf_counter()
# The input is the first state: only the key request; the nodes fill in the rest.
chain_result = trip_chain.invoke({"request": USER_MESSAGE_GOAL})
chain_seconds = time.perf_counter() - start

display(Markdown("**ANSWER**\n\n" + chain_result["answer"]))
print(f"linear workflow: {chain_seconds:.0f} s")

**ANSWER**

**Prague.** Both trips fit the 300 EUR budget, but Prague's bus takes about 4.5 hours each way versus 9.5 for Vienna, so you get more time there.
**Dates:** Saturday 7 to Sunday 8 November 2026, 1 night.
**Travel:** Return bus at 40 EUR per person (Vienna: 70 EUR).
**Money left per person:** 85 EUR for food and activities after bus and hostel (25 EUR per night). Vienna would leave only 45 EUR.
**Free:** Charles Bridge early in the morning and the view from Letna Park.
**Weather:** Saturday 2.1 to 6.1 °C with 4% rain chance; Sunday 3.1 to 7.2 °C with 13% rain chance. Pack a warm coat, layers, and an umbrella.
*All prices are course examples from the travel guide, not confirmed fares.*

linear workflow: 10 s


`chain_result` is the final state, with every key that a node wrote: `chain_result["facts"]` holds the facts of agent 1.

### One agent or a chain of agents?

| | One agent (`planner`, Step 5) | Linear workflow (`trip_chain`, Step 7) |
|---|---|---|
| Who chooses the tool calls | the model | the model, inside agent 1 |
| Who fixes the order of the steps | the model | our code, through the edges |
| Jobs per agent | collect facts and write the plan | one job each |
| Between the steps | nothing to check | our code can read and check the facts |
| Agent runs | one | two, one after the other |

So Step 7 is a **workflow whose steps are agents**. A chain is worth its extra agent run when the steps are clear in advance and a check between them helps. In both, the cities are still researched **one after the other**: Step 8 changes that.

## Step 8: a team of agents

### Why a team

Agent 1 of the chain researches Prague and Vienna one after the other, and so does the planner of Step 5. The two cities do not depend on each other, so two agents could research them **at the same time**.

The pattern for this has three roles, and each one is an agent built with `create_agent`:

- The **orchestrator agent** reads the request and lists the cities to compare.
- One **city agent** per city researches it with our four tools.
- The **synthesiser agent** compares the reports and recommends one city.

Anthropic names this pattern *orchestrator-workers*; in our graph every worker is an agent, so we call them city agents. Our code still fixes the order: orchestrator, then city agents, then synthesiser. So the team is a **workflow**, like the chain. The difference is the middle step: it runs in parallel.

```mermaid
graph LR;
    request[Request] --> orchestrator[Orchestrator agent: lists the cities];
    orchestrator --> prague[City agent: Prague];
    orchestrator --> vienna[City agent: Vienna];
    prague --> synthesiser[Synthesiser agent: compare and recommend];
    vienna --> synthesiser;
```

Running nodes in parallel needs two new graph ideas. We learn each one on a small graph without agents, so the cells run in a moment and cost nothing: first the **reducer**, then **`Send`**.

### Lesson 1: parallel nodes and a reducer

Two edges that leave the same node start two nodes **in the same step**: LangGraph runs them at the same time. Below, `START` leads to `prague` and to `vienna`, and both write to the same field `reports`.

**Predict.** Both nodes return a value for `reports` in the same step. Which value should the state keep?

In [ ]:
import operator
from typing import Annotated

from langgraph.errors import InvalidUpdateError


# Two nodes that write to the same field. No agents: each returns a fixed report.
def prague(state) -> dict:
    return {"reports": ["Prague report"]}


def vienna(state) -> dict:
    return {"reports": ["Vienna report"]}


def build_two_cities(state_class):
    """A graph where START leads to prague and to vienna, so both run in the same step."""
    builder = StateGraph(state_class)
    builder.add_node("prague", prague)
    builder.add_node("vienna", vienna)
    builder.add_edge(START, "prague")
    builder.add_edge(START, "vienna")
    builder.add_edge("prague", END)
    builder.add_edge("vienna", END)
    return builder.compile()


# 1. Without a reducer: a field holds one value, and two values arrive at once.
class StateWithoutReducer(BaseModel):
    reports: list[str] = []


try:
    build_two_cities(StateWithoutReducer).invoke({})
except InvalidUpdateError as error:
    print("Without a reducer:", str(error).splitlines()[0])


# 2. With a reducer: operator.add joins the two lists, as [a] + [b] does.
class StateWithReducer(BaseModel):
    reports: Annotated[list[str], operator.add] = []


print("With the reducer:  ", build_two_cities(StateWithReducer).invoke({})["reports"])

Without a reducer: At key 'reports': Can receive only one value per step. Use an Annotated key to handle multiple values.
With the reducer:   ['Prague report', 'Vienna report']


- **Without a reducer**, a field holds one value. When one node writes, the new value replaces the old one. When two nodes write in the same step, LangGraph cannot know which value to keep, so it stops with `InvalidUpdateError`.
- **With a reducer**, LangGraph combines the values. `Annotated[list[str], operator.add]` attaches the function `operator.add` to the field: for two lists, `operator.add(a, b)` is `a + b`. So each node returns a list with **one** report, and the state ends with a list of all of them.

This graph already answers a common question: **if you have a fixed set of agents A, B and C, you do not need `Send`.** Add an edge from the same node to each of them, and give the field they share a reducer. All three run in the same step.

### Lesson 2: `Send`, one branch per item

Fixed edges need the branches when we write the graph. Our cities are only known **at run time**: the orchestrator reads them from the request, and the next request may name three cities, or one. We also need the **same** node once per city, each time with a different city.

`Send(node_name, payload)` from `langgraph.types` starts one run of a node with its own input, the **payload**. A function returns a list of them, one per city; all of them run in the same step. The node receives the payload, not the whole state.

This function decides what runs next while the graph runs, so it does not fit on a normal edge: `add_edge(a, b)` always leads to `b`, exactly once. It goes on a **conditional edge**: `add_conditional_edges(source, function, [node names])`. LangGraph calls the function after `source` and starts what it returns. The list names the nodes the function **may** start; LangGraph uses it to check and draw the graph. It is not a list of nodes that must run: the function chooses, on every run, which of them start, how often, and whether any start at all.

**Predict.** The function below returns one `Send` per city. What happens for two cities, for three, and for an empty list?

In [ ]:
from langgraph.types import Send


class CityListState(BaseModel):
    cities: list[str]
    reports: Annotated[list[str], operator.add] = []


# The payload of one Send: the input of one run of measure_city.
class CityName(BaseModel):
    city: str


# The function on the conditional edge: one Send per city in the state.
def one_send_per_city(state: CityListState) -> list[Send]:
    return [Send("measure_city", CityName(city=city)) for city in state.cities]


# Runs once per Send; job is the payload, not the whole state.
def measure_city(job: CityName) -> dict:
    return {"reports": [f"{job.city} has {len(job.city)} letters"]}


demo_builder = StateGraph(CityListState)
demo_builder.add_node("measure_city", measure_city)
# After START, one_send_per_city decides what runs; it may only start measure_city.
demo_builder.add_conditional_edges(START, one_send_per_city, ["measure_city"])
demo_builder.add_edge("measure_city", END)
send_demo = demo_builder.compile()

for cities in [["Prague", "Vienna"], ["Prague", "Vienna", "Dresden"], []]:
    print(cities, "->", send_demo.invoke({"cities": cities})["reports"])

['Prague', 'Vienna'] -> ['Prague has 6 letters', 'Vienna has 6 letters']
['Prague', 'Vienna', 'Dresden'] -> ['Prague has 6 letters', 'Vienna has 6 letters', 'Dresden has 7 letters']
[] -> []


The same node ran twice, then three times, and for the empty list it did not run at all: the function may return any number of `Send`, also zero. The reducer from Lesson 1 collected the reports, because the runs of `measure_city` write to `reports` in the same step.

| What you know when you write the graph | What to use |
|---|---|
| The order of the steps: A, then B | `add_edge("A", "B")`, as in Step 7 |
| A fixed set of nodes that run together: A, B and C | an edge from the same node to each, and a reducer on the field they share (Lesson 1) |
| One of several nodes, chosen at run time | `add_conditional_edges` with a function that returns a node name |
| The same node once per item, and the items are known only at run time | `add_conditional_edges` with a function that returns a list of `Send` (Lesson 2) |

### The list says what may run, not what must run

In the demo above the list held one node, `measure_city`. A conditional edge can name several nodes, and the function picks among them for every item, or picks none. Below, the list names two research nodes, a quick one and a deep one, and the function decides per city:

- a city with a note in our travel guide gets `research_city_quick`: the prices are already known;
- a city without a note gets `research_city_deep`: an agent would have to search more;
- Berlin is where we live, so it gets no node at all.

**Predict.** Which nodes run for `["Prague", "Leipzig", "Berlin"]`, and which for `["Berlin"]`?

In [ ]:
# The function on the conditional edge: for each city, one Send to the quick node, one to the deep node, or none.
def choose_research(state: CityListState) -> list[Send]:
    sends = []
    for city in state.cities:
        if city == "Berlin":
            continue  # our home city: no node starts for it
        if any(note.startswith(city) for note in TRIP_GUIDE):
            sends.append(Send("research_city_quick", CityName(city=city)))
        else:
            sends.append(Send("research_city_deep", CityName(city=city)))
    return sends


def research_city_quick(job: CityName) -> dict:
    return {"reports": [f"{job.city}: quick check, the guide has a note"]}


def research_city_deep(job: CityName) -> dict:
    return {"reports": [f"{job.city}: deep research, the guide has no note"]}


choice_builder = StateGraph(CityListState)
choice_builder.add_node("research_city_quick", research_city_quick)
choice_builder.add_node("research_city_deep", research_city_deep)
# The list names the two nodes that choose_research MAY start; it may also start none.
choice_builder.add_conditional_edges(START, choose_research, ["research_city_quick", "research_city_deep"])
choice_builder.add_edge("research_city_quick", END)
choice_builder.add_edge("research_city_deep", END)
choice_demo = choice_builder.compile()

for cities in [["Prague", "Leipzig", "Berlin"], ["Berlin"]]:
    print(cities, "->", choice_demo.invoke({"cities": cities})["reports"])

['Prague', 'Leipzig', 'Berlin'] -> ['Prague: quick check, the guide has a note', 'Leipzig: deep research, the guide has no note']
['Berlin'] -> []


For Berlin no node ran, and for `["Berlin"]` the graph ended with no report at all. In our team, `fan_out` lists only `research_city`, and it starts one per city. A team for a real project could list `research_city_quick` and `research_city_deep`, or a router could skip the research when the request names no city. The edge only fixes **which nodes can follow**; the function decides **which do**.

### The three agents

All three agents in one cell, each with its rules written out:

- The **orchestrator agent** has **no tools**: it does not research anything. Its only job is to turn the sentence of the user into data that our code can loop over. `response_format=TripPlan` makes its final answer fit a **schema**: a Pydantic class whose fields have a type and a description, which the model reads like a docstring. The agent returns the answer as a `TripPlan` object under the key `"structured_response"`, so `plan.cities` is a Python list.
- The **city agent** has our four tools and researches one city.
- The **synthesiser agent** has no tools: it reads the reports and recommends one city.

In [ ]:
from pydantic import Field


# The schema of the orchestrator's answer: each field has a type and a description, which the model reads.
class TripPlan(BaseModel):
    """A trip request split into one task per city."""

    people: int = Field(description="Number of travellers")
    budget_eur: float = Field(description="Total budget of the whole group in euros")
    nights: int = Field(description="Number of nights; 1 for a weekend")
    cities: list[str] = Field(description="The destination cities the user mentions, in English")


ORCHESTRATOR_RULES = """You read a trip request and fill in the plan: the number of people, the total budget,
the number of nights and the cities to compare. You do not research anything."""

CITY_RULES = """You research one city for a short, cheap trip from Berlin.
Rules:
- Research only the one city you are given.
- Use the tools for every fact: dates, weather, fares, prices and sums. Never guess a number.
- Prices from the travel guide are course examples. Say so in your report.
- Report in at most 8 lines: the dates, the travel, the bed, the money left per person, the weather and what to pack.
- Do not compare with other cities."""

SYNTHESISER_RULES = """You compare trip options for students who live in Berlin.
Use only the facts in the reports. Recommend one city in at most 6 lines."""

# 1. Orchestrator: no tools; response_format makes its final answer a TripPlan object.
orchestrator_agent = create_agent(model=model, tools=[], system_prompt=ORCHESTRATOR_RULES, response_format=TripPlan)
# 2. City agent: the four tools of Part 1, for one city.
city_agent = create_agent(model=model,
                          tools=[next_weekend, weather_forecast, search_trip_guide, budget_per_person],
                          system_prompt=CITY_RULES)
# 3. Synthesiser: no tools; it reads the reports we pass on.
synthesiser_agent = create_agent(model=model, tools=[], system_prompt=SYNTHESISER_RULES)

### The team graph

The team uses both lessons: `fan_out` sends one `CityJob` per city to `research_city` (Lesson 2), and the reducer on `reports` collects the reports of the city agents (Lesson 1).

```mermaid
graph TD;
    start_node([START: request = USER_MESSAGE_GOAL]) --> orchestrator_node[orchestrator: orchestrator_agent, response_format TripPlan];
    orchestrator_node -- writes plan: TripPlan --> fan_out_node{fan_out: one Send per city};
    fan_out_node -- "Send CityJob Prague" --> agent_prague;
    fan_out_node -- "Send CityJob Vienna" --> agent_vienna;
    subgraph agent_prague [research_city: city_agent on Prague]
        model_p[Model call] --> tools_p[Tools] --> model_p;
    end
    subgraph agent_vienna [research_city: city_agent on Vienna]
        model_v[Model call] --> tools_v[Tools] --> model_v;
    end
    agent_prague -- "reports: [Prague report]" --> reducer_node[reducer operator.add: one list of reports];
    agent_vienna -- "reports: [Vienna report]" --> reducer_node;
    reducer_node --> synthesiser_node[synthesiser: synthesiser_agent, no tools];
    synthesiser_node -- writes recommendation --> end_node([END]);
```

In [ ]:
class TeamState(BaseModel):
    request: str
    plan: TripPlan | None = None
    # The reducer: the city agents write here in the same step (Lesson 1).
    reports: Annotated[list[str], operator.add] = []
    recommendation: str = ""


# The payload of one Send: the city and the plan for one city agent (Lesson 2).
class CityJob(BaseModel):
    city: str
    plan: TripPlan


# Node 1: the orchestrator agent turns the request into a TripPlan.
def orchestrator(state: TeamState) -> dict:
    """Split the request into one task per city."""
    result = orchestrator_agent.invoke({"messages": [("user", state.request)]})
    return {"plan": result["structured_response"]}


# On the conditional edge, not a node: one Send per city in the plan.
def fan_out(state: TeamState) -> list[Send]:
    """Start one city agent per city in the plan."""
    return [Send("research_city", CityJob(city=city, plan=state.plan)) for city in state.plan.cities]


# Node 2, once per city: job is the CityJob given by Send, not the whole state.
def research_city(job: CityJob) -> dict:
    """Research one city with the city agent and return its report."""
    plan = job.plan
    brief = (f"Research {job.city} for this weekend: from Berlin, {plan.people} people, "
             f"{plan.budget_eur} EUR in total, {plan.nights} night(s).")
    result = city_agent.invoke({"messages": [("user", brief)]})
    # Return a list with ONE report. All city agents return in the same step, and the
    # reducer operator.add on TeamState.reports joins their lists into one list.
    # Without the reducer, LangGraph would stop here with InvalidUpdateError (Lesson 1).
    return {"reports": [f"## {job.city}\n{result['messages'][-1].text}"]}


# Node 3: the synthesiser agent reads all reports and recommends one city.
def synthesiser(state: TeamState) -> dict:
    """Compare the reports and recommend one city."""
    reports = "\n\n".join(state.reports)
    result = synthesiser_agent.invoke({"messages": [("user", f"Request: {state.request}\n\nReports:\n{reports}")]})
    return {"recommendation": result["messages"][-1].text}

### Wiring the team

The wiring is the same as for the chain, with the conditional edge from Lesson 2: after the orchestrator, `fan_out` decides how many city agents run. The list `["research_city"]` names the nodes `fan_out` may start; with more research nodes in the list, it could choose among them, and with no cities in the plan it starts none. `research_city` has a normal edge to the synthesiser, so the synthesiser waits until every city agent has finished.

In [30]:
team_builder = StateGraph(TeamState)
team_builder.add_node("orchestrator", orchestrator)
team_builder.add_node("research_city", research_city)
team_builder.add_node("synthesiser", synthesiser)
team_builder.add_edge(START, "orchestrator")
# fan_out decides what runs next. The list names the nodes it MAY start, not the ones it must start:
# it could also name ["research_city_quick", "research_city_deep"] and choose per city, or start none.
team_builder.add_conditional_edges("orchestrator", fan_out, ["research_city"])
# The synthesiser waits until every city agent has finished.
team_builder.add_edge("research_city", "synthesiser")
team_builder.add_edge("synthesiser", END)
trip_team = team_builder.compile()

**Predict.** Which finishes first: the single `planner` of Step 5, the chain of Step 7 or the team? The cell times the team and the planner; the chain's time is still in `chain_seconds`.

In [31]:
start = time.perf_counter()
# Time the team ...
team_result = trip_team.invoke({"request": USER_MESSAGE_GOAL})
team_seconds = time.perf_counter() - start

start = time.perf_counter()
# ... and the single planner of Step 5 on the same question.
planner.invoke({"messages": [("user", USER_MESSAGE_GOAL)]})
single_seconds = time.perf_counter() - start

for report in team_result["reports"]:
    display(Markdown(report))
display(Markdown("## Recommendation\n\n" + team_result["recommendation"]))
print(f"team: {team_seconds:.0f} s, linear workflow: {chain_seconds:.0f} s, single agent: {single_seconds:.0f} s")

## Prague
**Prague, Czechia, for Saturday 7 to Sunday 8 November 2026 (1 night).** Prague is the cheapest city in the guide and the trip fits your budget.
**Travel:** Bus from Berlin, about 4.5 hours each way, about 40 EUR return per person (guide example price).
**Bed:** Hostel, about 25 EUR per night (guide example price).
**Money:** Of 150 EUR per person, 65 EUR goes to fare and bed, leaving **85 EUR per person** for food and activities.
**Food and free things:** A lunch menu costs about 8 EUR (guide example). Charles Bridge early in the morning and the view from Letna Park are free.
**Weather:** Highs of 6 °C on Saturday and 7 °C on Sunday, lows of 2 to 3 °C, rain chance 4% Saturday and 13% Sunday.
**Pack:** A warm coat, layers, a hat and gloves, and comfortable shoes for walking. A small umbrella is optional.
All prices above come from the travel guide's course examples, so check current fares and beds before you book.

## Vienna
- **City:** Vienna, Austria. It fits the budget, and the guide lists a free Saturday-morning flea market at the Naschmarkt.
- **Dates:** Sat 7 Nov to Sun 8 Nov 2026, 1 night.
- **Travel:** Bus from Berlin, about 9.5 hours, roughly 70 EUR return per person (course example from the travel guide).
- **Bed:** Hostel, about 35 EUR per night (course example from the travel guide).
- **Money left:** 45 EUR per person for food and activities, out of 150 EUR each. The whole trip is within the 300 EUR budget.
- **Weather:** Sat 2–9.3 °C, 3% rain. Sun 0.8–7.8 °C, 5% rain. Dry but cold.
- **Pack:** Warm coat, hat, gloves, scarf, and comfortable shoes for walking.

## Recommendation

**Recommendation: Prague.** It leaves about 85 EUR per person for food and activities, versus 45 EUR in Vienna. The bus takes about 4.5 hours each way, compared with about 9.5 in Vienna. Charles Bridge and Letna Park views are free, and the weather is milder. The prices come from the guide's examples, so check current fares and hostel rates before booking.

team: 9 s, linear workflow: 10 s, single agent: 7 s


### When is a team worth it?

The city agents ran at the same time, so the team waits about as long as the slowest city agent, plus the orchestrator agent and the synthesiser agent. But the team runs the most agents: one orchestrator, one per city, one synthesiser. Network and model times vary, so run the cells twice before you conclude.

A simple guideline, in this order:

1. Start with **one agent**, as long as one conversation can hold the task.
2. Split it into a **chain of agents** when the steps are clear in advance and a check between them helps.
3. Move to a **team of agents** when the task splits into **independent** parts (one per city, per document, per product), when there are several of them, and when the waiting time matters. With one city, a team only adds agent runs.

**See it run.** The folder `trip_planner_app` runs this team as a web page: it draws the graph, lights up each node while it runs, shows the city agents in parallel and the reducer joining their reports, and pauses with `interrupt()` to ask you when the request is unclear. See [its README](trip_planner_app/README.md).

## Check your understanding

Vote alone first, then discuss with your neighbour, then vote again.

**1.** In Step 1, the agent described the weather for the weekend. Where did that come from?

- A. From a weather service that the model calls by itself.
- B. From the model's training data and guessing: it had no tool and does not know today's date.
- C. From the system prompt.
- D. From LangChain, which adds the weather to every request.

**2.** Who runs the Python function of a tool?

- A. Claude, on Anthropic's servers.
- B. Our own Python process: the model only asks for the call.
- C. The web service of the tool.
- D. Nobody; the model imagines the result.

**3.** Why does the budget come from the tool `budget_per_person` and not from the model?

- A. The model is not allowed to see numbers.
- B. A sum in Python is exact and the same every time; a model can make arithmetic mistakes.
- C. Tools are cheaper than tokens.
- D. The system prompt cannot mention money.

**4.** `planner` does not understand the follow-up question, but `memory_planner` does. What is the difference?

- A. `memory_planner` uses a larger model.
- B. `memory_planner` has a checkpointer, and both calls use the same `thread_id`, so the first conversation is saved and read again.
- C. `memory_planner` has more tools.
- D. The follow-up was asked twice.

**5.** In the chain of Step 7, `writer_agent` has no tools. Where do the numbers in its plan come from?

- A. From the model's training data.
- B. From the facts of `research_agent`, which our code passes on in the user message.
- C. `writer_agent` calls the tools of `research_agent`.
- D. From the system prompt `WRITER_RULES`.

**6.** The team of agents made more model calls than the single planner but can finish sooner. Why?

- A. Haiku answers faster inside a graph.
- B. The city agents ran at the same time, so the waiting time follows the slowest city agent, not the sum.
- C. `Send` saves results so later calls are skipped.
- D. The synthesiser has no tools.

## Exercises

Work in pairs. Every exercise needs your API key.

### 5.1 · Predict the trajectory

*Predict*

> Without running anything, write down which tools `planner` will call for the question *"What should I pack for Prague this weekend?"*, in which order, and which tools it will not need. Then run it with `show_run` and compare. Where did the model surprise you?

*Source:* course-written for this workbook.

*Your answer:*

### 5.2 · Add a city to the knowledge base

*Code*

> Add a note for Leipzig to `TRIP_GUIDE`, in the same form as the others (your own example prices). Rebuild `planner` and ask: *"Leipzig or Dresden this weekend, 2 people, 150 EUR?"* Check in the `tool returns` lines that the agent found your note. Then remove the hostel price from your note and ask again: does the agent say that the price is missing, or does it guess?

*Source:* course-written for this workbook.

### 5.3 · A currency tool with a second web service

*Code*

> Prague uses the Czech koruna. Write a tool `convert_currency(amount: float, from_currency: str, to_currency: str) -> str` with the free Frankfurter service: `httpx.get("https://api.frankfurter.dev/v2/rates", params={"base": "EUR", "quotes": "CZK"})` returns a list such as `[{"date": "...", "base": "EUR", "quote": "CZK", "rate": ...}]`. Return the converted amount, the rate and its date, and an "Error: ..." sentence if the request fails. Test it by hand, add it to the tools, and ask: *"How much is 85 EUR in Prague money?"*

*Source:* course-written for this workbook; API: [Frankfurter](https://frankfurter.dev/).

### 5.4 · A question the tools cannot answer

*Predict*

> Ask `planner` about a weekend more than 16 days from today, for example *"We want to go to Vienna in four weeks. What should we pack?"* The weather tool will return an error. Predict first: will the agent stop, try again, or answer without a forecast? Then run it. Is the answer honest about what it does not know?

*Source:* course-written for this workbook.

### 5.5 · A check between the two agents

*Code*

> A chain lets our code check the work of agent 1 before agent 2 starts. Write a function `facts_look_complete(facts: str) -> bool` that returns `True` if the facts name every city of the request and contain "EUR". Run `research_agent`, then run `writer_agent` only if the check passes; otherwise print the facts. Which mistakes of agent 1 does your check catch, and which does it miss?

*Source:* course-written for this workbook.

### 5.6 · A bigger team

*Pairs*

> Change `USER_MESSAGE_GOAL` to three cities: Prague, Vienna or Dresden. Run the chain, the team and the single planner again and note the three times. Which gains more from the extra city? Write two sentences of advice for a team that must choose between one agent and a team of agents.

*Source:* course-written for this workbook.

## Debrief

- **Workflow or agent?** For Steps 5, 7 and 8, say who chose the next tool call: our code or the model.
- **Compare the answers of Step 1 and Step 5** for the same question: list each number in the Step 5 answer and the tool it came from.
- **Compare two pairs' trajectories** for exercise 5.1. They are usually not the same: an agent is not deterministic.
- **Exercise 5.2:** which pairs saw the agent guess a missing price? A knowledge base helps only if the agent says when it finds nothing.
- **Exercise 5.5:** a check in code between two agents always runs; a rule in a prompt may be ignored. Which checks of your project belong in code?
- **Exercise 5.6:** a choice of architecture should quote measured times, not a general preference.

## Further reading

- LangChain docs: [Agents](https://docs.langchain.com/oss/python/langchain/agents), [Tools](https://docs.langchain.com/oss/python/langchain/tools), [Short-term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory), [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output).
- LangGraph docs: [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents), sections *Prompt chaining* (Step 7) and *Orchestrator-worker*, the source of the `Send` pattern in Step 8.
- Anthropic: [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents), on workflows, prompt chaining, orchestrator-workers and when an agent is worth it; [Pricing](https://platform.claude.com/docs/en/about-claude/pricing), read on 3 November 2026.
- Notebooks 01, 03 and 04 of this session: retrieval with embeddings, the LangChain syntax in full, and more multi-agent patterns.
- Open-Meteo: [forecast docs](https://open-meteo.com/en/docs), [geocoding docs](https://open-meteo.com/en/docs/geocoding-api), [terms](https://open-meteo.com/en/terms) (free for non-commercial use, data under CC BY 4.0).
- Frankfurter (exercise 5.3): [docs](https://frankfurter.dev/), [licence and data providers](https://frankfurter.dev/license/).